In [1]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from docplex.mp.dvar import Var
from docplex.mp.model import Model

Arco = Tuple[str, str]


def ler_instancia_pfcm(
    caminho: Path,
) -> Tuple[Dict[str, Tuple[str, int]], Dict[Arco, Tuple[float, Optional[float]]]]:
    linhas = [
        linha.strip()
        for linha in caminho.read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    n_nos, n_arcos = map(int, linhas[0].split())

    nos: Dict[str, Tuple[str, int]] = {}
    for linha in linhas[1 : 1 + n_nos]:
        nome, tipo, valor = linha.split()
        nos[nome] = (tipo.upper(), int(valor))

    arcos: Dict[Arco, Tuple[float, Optional[float]]] = {}
    for linha in linhas[1 + n_nos : 1 + n_nos + n_arcos]:
        u, v, custo_str, cap_str = linha.split()
        cap: Optional[float] = None if cap_str.upper() == "INF" else float(cap_str)
        arcos[(u, v)] = (float(custo_str), cap)

    return nos, arcos


nos, arcos = ler_instancia_pfcm(Path("in.txt"))
print(f"Instância carregada: {len(nos)} nós e {len(arcos)} arcos.")

Instância carregada: 9 nós e 14 arcos.


In [2]:
mdl = Model(name="PFCM")

# Variáveis inteiras x_ij >= 0 com limite superior u_ij (quando finito) e l_ij = 0
x: Dict[Arco, Var] = {}
for (u, v), (_, cap) in arcos.items():
    if cap is None:
        x[(u, v)] = mdl.integer_var(lb=0, name=f"x_{u}{v}")
    else:
        x[(u, v)] = mdl.integer_var(lb=0, ub=cap, name=f"x_{u}{v}")

# (1) Função Objetivo: minimizar o custo total de transporte
mdl.minimize(mdl.sum(custo * x[(u, v)] for (u, v), (custo, _) in arcos.items()))

# (2), (3) e (4) Restrições de oferta (S), demanda (D) e transbordo (T)
for no, (tipo, valor) in nos.items():
    fluxo_saida = mdl.sum(x[(u, v)] for (u, v) in arcos if u == no)
    fluxo_entrada = mdl.sum(x[(u, v)] for (u, v) in arcos if v == no)

    if tipo == "S":
        mdl.add_constraint(fluxo_saida - fluxo_entrada <= valor, ctname=f"Orig_{no}")
    elif tipo == "D":
        mdl.add_constraint(fluxo_entrada - fluxo_saida >= valor, ctname=f"Dest_{no}")
    elif tipo == "T":
        mdl.add_constraint(fluxo_saida - fluxo_entrada == 0, ctname=f"Tran_{no}")
    else:
        raise ValueError(f"Tipo de nó desconhecido: {tipo}")

print("Modelo PFCM construído com sucesso.")

Modelo PFCM construído com sucesso.


In [3]:
sol = mdl.solve(log_output=False)

if sol is None:
    print("Nenhuma solução encontrada.")
else:
    print(f"Status: {mdl.solve_details.status}")
    print(f"Função Objetivo (Custo Mínimo): {sol.objective_value:.2f}\n")
    print("Fluxo ótimo por arco:")
    for (u, v), (custo, cap) in arcos.items():
        val = int(round(sol.get_value(x[(u, v)])))
        cap_txt = "INF" if cap is None else f"{cap:.0f}"
        print(f"  x[{u}->{v}] = {val:>2d}   (custo unit.: {custo:.0f}, cap.: {cap_txt})")

Status: integer optimal solution
Função Objetivo (Custo Mínimo): 184.00

Fluxo ótimo por arco:
  x[F->P] =  2   (custo unit.: 7, cap.: INF)
  x[F->M] =  8   (custo unit.: 2, cap.: 8)
  x[G->M] =  7   (custo unit.: 5, cap.: 7)
  x[G->H] =  3   (custo unit.: 2, cap.: INF)
  x[H->M] =  4   (custo unit.: 6, cap.: INF)
  x[H->N] =  3   (custo unit.: 5, cap.: 9)
  x[H->S] =  6   (custo unit.: 4, cap.: 6)
  x[M->P] = 16   (custo unit.: 1, cap.: 17)
  x[M->Q] =  3   (custo unit.: 3, cap.: 4)
  x[M->R] =  0   (custo unit.: 4, cap.: INF)
  x[N->S] =  3   (custo unit.: 3, cap.: INF)
  x[P->Q] = 10   (custo unit.: 1, cap.: 10)
  x[Q->R] =  6   (custo unit.: 1, cap.: INF)
  x[R->N] =  0   (custo unit.: 2, cap.: INF)
